# Lab 7 — Streaming Responses with SSE for AI Chat

**Difficulty: Intermediate | ~40 min | Requires Lab 2 (Async/Await)**

### Step 0: Install Dependencies

This cell installs every pinned dependency the lab needs. `uvicorn` is the ASGI server used to serve the app in the demos — the one new dependency over earlier labs, and it is required because (as Step 5 explains) FastAPI's `TestClient` cannot expose streaming chunks incrementally.

In [1]:
!pip install fastapi==0.112.2 pydantic==2.8.2 httpx==0.28.1 python-dotenv==1.2.3 openai==3.5.0 uvicorn==0.30.6


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Step 1: Imports, API Key, and App Setup

The imports give us FastAPI and `StreamingResponse` for the two endpoints, `AsyncOpenAI` for the OpenRouter LLM client, `httpx` for the HTTP client that consumes the stream, `uvicorn` to serve the app, and `threading`/`time` to run the server in a background thread and measure timing. The API key is read from the `.env` file with an input fallback.

The demo prompt used throughout produces a reasonably long answer (about 100 words). That length is deliberate: a short answer finishes so fast that the timing difference between blocking and streaming is too small to see, which is precisely the effect this lab measures.

In [2]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from dotenv import load_dotenv
from openai import AsyncOpenAI
import httpx, uvicorn
import os, time, threading

load_dotenv()

api_key = os.getenv("OPEN_ROUTER_KEY")
if not api_key:
    api_key = input("Open Router API key: ")

client = AsyncOpenAI(
    api_key=api_key,
    base_url="https://openrouter.ai/api/v1"
)

app = FastAPI()

### Step 2: Request Model

Both endpoints share a simple Pydantic model for the request body. `message` is the user input; `simulate_error` is only meaningful for the streaming endpoint and defaults to `False`.

In [3]:
class ChatMessage(BaseModel):
    message: str
    simulate_error: bool = False

### Step 3: The `/chat/blocking` Endpoint

A standard non-streaming endpoint: send the user message to OpenRouter, `await` the full completion, and return the entire answer as a single JSON object. This is the baseline the timing comparison measures against — the client cannot see any part of the answer until OpenRouter has finished generating the whole thing.

In [4]:
@app.post("/chat/blocking")
async def chat_blocking(msg: ChatMessage):
    response = await client.chat.completions.create(
        model="openrouter/free",
        messages=[{"role": "user", "content": msg.message}],
    )
    return {"answer": response.choices[0].message.content}

### Step 4: The Streaming Generator and `/chat/stream` Endpoint

This is the core of the lab. `stream_tokens` is an async generator function — the `yield` keyword makes it one. It opens a streaming connection to OpenRouter by passing `stream=True`, then iterates the returned async iterator chunk by chunk.

For each chunk it extracts `delta.content` — the next piece of text the LLM just produced; typically a word or a small phrase, sometimes a larger span — and yields it as a properly formatted **Server-Sent Event**: an `event: token` line, a `data:` line carrying the text, and a blank line that ends the event. 

OpenRouter occasionally sends keep-alive chunks whose content is empty or `None`; the `if delta.content:` check skips those silently instead of turning them into fake token events.

When `simulate_error=True`, the generator yields exactly 2 real chunks, then yields an error event and returns. The `try/finally` wrapper closes the upstream stream even when the generator returns early, so the LLM connection is never left hanging. Because the error is raised *from inside the generator*, the client receives a clean `event: error` and the response ends normally — there is no crash and no hanging connection (see Section 7 for why this cannot be done with a `try/except` around the endpoint).

In [5]:
async def stream_tokens(message: str, simulate_error: bool = False):
    stream = await client.chat.completions.create(
        model="openrouter/free",
        messages=[{"role": "user", "content": message}],
        stream=True,
    )
    event_count = 0
    try:
        async for chunk in stream:
            delta = chunk.choices[0].delta
            if delta.content:
                if simulate_error and event_count >= 2:
                    yield "event: error\ndata: Stream interrupted after partial response\n\n"
                    return
                yield f"event: token\ndata: {delta.content}\n\n"
                event_count += 1
    finally:
        await stream.close()
    yield "event: done\ndata: [DONE]\n\n"

@app.post("/chat/stream")
async def chat_stream(msg: ChatMessage):
    return StreamingResponse(
        stream_tokens(msg.message, msg.simulate_error),
        media_type="text/event-stream",
    )

### Step 5: Run the App on a Real Server

So far, the app is just an object. To measure streaming properly, we’ll run it as a real HTTP server using `Uvicorn` and connect to it with a real `httpx.Client`.

This matters because the lab is specifically measuring when the first piece of output arrives. With a real HTTP connection, each SSE chunk reaches the client as soon as the server yields it, allowing us to measure that first chunk accurately.

We’ll therefore run `Uvicorn` in a background thread and use `httpx.Client` to consume the stream. This gives us the end-to-end behavior we actually want to observe: the server produces a chunk → the chunk travels over HTTP → the client receives it → we record the timing.

For comparison, FastAPI’s `TestClient` completes the request through its in-process transport before returning the response, so streamed output is effectively buffered from the perspective of our timing measurement. It’s useful for testing correctness, but not for measuring real streaming latency.

**Note:** because the server thread runs its own async event loop, you may very occasionally see an "event loop is closed" warning appear. If any cell errors unexpectedly, simply re-run that cell — the server recovers on retry.

In [6]:
PORT = 8777


def run_server():
    uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)

http_client = httpx.Client(timeout=180)
print("Server up:", http_client.get(f"http://127.0.0.1:{PORT}/").status_code)  # 404 expected — no root route

Server up: 404


### Demo 1: Blocking `/chat/blocking` — the baseline

A normal call to the blocking endpoint through the real server. The client sends the message and receives a complete JSON response only after the LLM has finished generating the entire answer. The elapsed time is the full round trip — the client sees nothing until this moment.

In [ ]:
start = time.perf_counter()
res = http_client.post(
    f"http://127.0.0.1:{PORT}/chat/blocking",
    json={
        "message": 'Explain how rainbows form. Cover refraction, reflection, dispersion, and why each observer sees their own rainbow, in about 100 words.'},
)
blocking_elapsed = time.perf_counter() - start

print(f"Blocking response ({blocking_elapsed:.1f}s):")
print(res.json()["answer"])

Blocking response (5.0s):
Rainbows form when sunlight enters a raindrop, slowing and bending as it passes from air into denser water (refraction). Because different wavelengths bend by different amounts, white light separates into its spectral colors (dispersion). The light then reflects off the back inner surface of the droplet and exits, bending again as it leaves. This combination produces a spectrum of colors at a specific angle—about 42° for red, 40° for violet—relative to the antisolar point (directly opposite the sun).

Each observer sees their own rainbow because the geometry depends on the viewer's position. A particular drop sends red light to your eye only if it lies along the exact 42° cone from your line of sight. Your neighbor's cone intersects different drops, so you see different suspended droplets producing your rainbow.


### Demo 2: Streaming `/chat/stream` — output arrives incrementally

We consume the stream with `http_client.stream()` — a context manager — rather than a plain `.post()`. A plain `.post()` would download the entire response body into memory before our code saw a single line, which would make a "time to first output" measurement meaningless. Inside the `with` block, `response.iter_lines()` yields each line of the SSE stream as it arrives over the real HTTP connection, and we process it immediately.

Each SSE event is one `data:` line carrying a chunk of the answer. The demo prints the first few chunks as they arrive (so you can see the incremental sequence), then prints the stitched-together answer and — the headline measurement — the time until the first chunk appeared.

In [7]:
start = time.perf_counter()
first_output_time = None
chunks_received = []

with http_client.stream(
    "POST",
    f"http://127.0.0.1:{PORT}/chat/stream",
    json={"message": 'Explain how rainbows form. Cover refraction, reflection, dispersion, and why each observer sees their own rainbow, in about 100 words.'},
) as response:
    for line in response.iter_lines():
        if line.startswith("data: ") and not line.startswith("data: [DONE]"):
            elapsed = time.perf_counter() - start
            if first_output_time is None:
                first_output_time = elapsed
                
            chunk = line[6:]
            chunks_received.append(chunk)

            if len(chunks_received) <= 5:
                print(f"  +{elapsed:.3f}s  {chunk!r}", flush=True)

print("\n")
print(f"Time to first output: {first_output_time:.3f}s")
print(f"Received {len(chunks_received)} chunks")
print(f"Stitched answer: {''.join(chunks_received)}")

  +3.416s  'A'
  +3.418s  ' rainbow forms'
  +3.420s  ' when sunlight enters'
  +3.422s  ' a rain'
  +3.426s  'drop, refracts'


Time to first output: 3.416s
Received 76 chunks
Stitched answer: A rainbow forms when sunlight enters a raindrop, refracts (bends) as it passes from air into water, then reflects off the drop's inner surface, and refracts again as it exits. Because different wavelengths of light bend by slightly different amounts—red least, violet most—the white light disperses into a spectrum of colors. The classic rainbow appears at about 42°, the deviation angle for red light, with violet at roughly 40°.Each observer sees their own rainbow because only raindrops positioned at the correct angle between the sun, drop, and your eye send light to you. A nearby person views light from a different set of drops, so every rainbow is unique to its viewer's perspective—rainbows lack a single physical existence in space.


### Demo 3: Side-by-side timing comparison

The total time to finish generating the full answer is roughly similar between the two approaches. What changes dramatically is **when the client sees the first piece of output** — and that is the difference between a user staring at a blank screen and a user immediately seeing a response begin to appear.

(The exact numbers vary from run to run — `openrouter/free` routes to different underlying models with different latencies. Run the notebook a couple of times and watch the relationship, not the precise values: for a long answer the streaming endpoint's first output almost always arrives well before the blocking endpoint finishes its round trip.)

In [9]:
print(f"Blocking: first output at {blocking_elapsed:.1f}s (full response arrives then)")
print(f"Streaming: first output at {first_output_time:.1f}s (then more chunks keep arriving)")
print()
print("Total generation time is roughly similar — streaming does not make the LLM finish faster.")
print("What streaming changes is how long the user waits before seeing anything at all.")

Blocking: first output at 5.9s (full response arrives then)
Streaming: first output at 1.2s (then more chunks keep arriving)

Total generation time is roughly similar — streaming does not make the LLM finish faster.
What streaming changes is how long the user waits before seeing anything at all.


### Demo 4: Mid-stream error simulation

To understand why this demo exists, first notice what has already reached the client the moment a streaming response begins: the HTTP status line, including status code 200. That status is committed before the LLM has generated anything, so if the stream fails partway through — and LLM streams do fail partway in production — it is far too late to send a different status code like 502. The failure therefore has to travel inside the stream itself, and the only tool left is the SSE format: yield an `event: error` line carrying a `data:` message, then stop. That is exactly what this demo teaches.

When `simulate_error=True`, the generator yields 2 real chunks from the LLM response, then yields `event: error` with the message `Stream interrupted after partial response` and ends. The client sees the partial answer, then a clear error signal, then a clean end — no crash, no hanging connection.

In [10]:
print("Simulating mid-stream error (2 real chunks then interruption):")
print("-" * 50)

with http_client.stream(
    "POST",
    f"http://127.0.0.1:{PORT}/chat/stream",
    json={"message": 'Explain how rainbows form. Cover refraction, reflection, dispersion, and why each observer sees their own rainbow, in about 100 words.', "simulate_error": True},
) as response:
    for line in response.iter_lines():
        if line.startswith("event:") or line.startswith("data:"):
            print(f"  {line}")

print("-" * 50)
print("Stream ended cleanly — no crash, no hang.")

Simulating mid-stream error (2 real chunks then interruption):
--------------------------------------------------
  event: token
  data: A
  event: token
  data:  rainbow forms
  event: error
  data: Stream interrupted after partial response
--------------------------------------------------
Stream ended cleanly — no crash, no hang.


The four demos illustrate the core points: blocking waits for the entire response, streaming delivers the first chunk almost immediately, the time-to-first-output is dramatically different even though total generation time is similar, and a mid-stream failure is handled gracefully inside the generator rather than crashing the endpoint.